# accel-sim silicon anchor — chain-depth isolation

The previous chain-grid notebook varied depth, per-layer shape, and token
count all at once across 4 configs — the isolated-vs-chained "discount"
factor didn't move monotonically with depth across them, so depth's effect
couldn't be separated from shape's.

This notebook holds shape (768×768, GPT-2's own projection dimension) and
token count (8192) **fixed** and varies **only depth** (1/2/4/6/8 identical
layers chained), each as one real forward+backward+Adam call. If a real
chained backward pays a roughly fixed per-chain overhead plus a per-layer
marginal cost, `measured_backward_ms` should come out close to linear in
depth — `compare_chain_depth.py` fits that line and reports R².

**Before running:** `Runtime > Change runtime type > T4 GPU`, then
`Runtime > Run all`.

Writes `chain_depth_profile.json`, prints it, and auto-downloads it. Bring
that file back and run:

```bash
python validate/silicon/compare_chain_depth.py chain_depth_profile.json
```


In [ ]:
# ---- config (edit if you want) ---------------------------------------------
ITERS  = 50
WARMUP = 15
OUT    = "chain_depth_profile.json"

SHAPE = (768, 768)   # square -- any depth composes trivially
DEPTHS = [1, 2, 4, 6, 8]
TOKENS = 8 * 1024


In [ ]:
import torch
assert torch.cuda.is_available(), "no CUDA device -- Runtime > Change runtime type > T4 GPU"

device = torch.device("cuda")
dtype = torch.float16
gpu = torch.cuda.get_device_name(0)
print(f"GPU: {gpu}   dtype=fp16   shape={SHAPE}   tokens={TOKENS}   iters={ITERS} (+{WARMUP} warmup)")


In [ ]:
import statistics
import torch.nn as nn

def bench(depth, M, iters, warmup):
    layers = [nn.Linear(*SHAPE, bias=False).to(device=device, dtype=dtype)
              for _ in range(depth)]
    params = [p for l in layers for p in l.parameters()]
    opt = torch.optim.Adam(params, lr=1e-4)
    x = torch.randn(M, SHAPE[0], device=device, dtype=dtype, requires_grad=True)

    fwd, bwd, optt = [], [], []
    for i in range(warmup + iters):
        ev = [torch.cuda.Event(enable_timing=True) for _ in range(5)]
        ev[0].record()
        out = x
        for l in layers:
            out = l(out)
        ev[1].record()
        loss = out.float().square().mean()
        ev[2].record()
        loss.backward()
        ev[3].record()
        opt.step()
        opt.zero_grad(set_to_none=True)
        x.grad = None
        ev[4].record()
        torch.cuda.synchronize()
        if i >= warmup:
            fwd.append(ev[0].elapsed_time(ev[2]))
            bwd.append(ev[2].elapsed_time(ev[3]))
            optt.append(ev[3].elapsed_time(ev[4]))

    def stat(v):
        return {"mean_ms": statistics.fmean(v),
                "std_ms": statistics.pstdev(v) if len(v) > 1 else 0.0}
    return {"forward": stat(fwd), "backward": stat(bwd), "optimizer": stat(optt)}


In [ ]:
results = {}
for depth in DEPTHS:
    r = bench(depth, TOKENS, ITERS, WARMUP)
    results[str(depth)] = r
    print(f"  depth {depth:2d}  fwd {r['forward']['mean_ms']:8.3f}  "
          f"bwd {r['backward']['mean_ms']:8.3f}  "
          f"opt {r['optimizer']['mean_ms']:6.3f} ms")


In [ ]:
import json, platform

out = {
    "gpu": gpu, "torch": torch.__version__, "cuda": torch.version.cuda,
    "dtype": "float16", "shape": SHAPE, "tokens": TOKENS, "depths": DEPTHS,
    "iters": ITERS, "warmup": WARMUP,
    "results": results, "host": platform.platform(),
}
with open(OUT, "w") as f:
    json.dump(out, f, indent=2)

print(f"\n===== {OUT} (copy this back if the download fails) =====\n")
print(json.dumps(out, indent=2))

try:
    from google.colab import files
    files.download(OUT)
except Exception as e:
    print(f"\n(auto-download unavailable: {e} -- grab {OUT} from the Files sidebar)")
